[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kkaitechnology-coder/free-python-for-development/blob/main/lesson-6-4-package-publishing-and-deployment.ipynb)

# Lesson 6.4 · Package Publishing & Deployment

**Free Python For Development** · Topic 6: Advanced Python & Real-World Projects

_setuptools, PyPI, Docker, cloud._

▶ **Watch this lesson with the AI voice tutor:** [open it in OratorLab](https://oratorlab.in/course-player.html?courseId=python-for-development&subtopicId=subtopic-6-4)

**How to use this notebook:** run each code cell with **Shift + Enter**, or use **Runtime → Run all**. Then change the code and run it again — experimenting is the fastest way to learn.

## Introduction & Packaging Your Python Project

Imagine you have just finished building an amazing Python tool and want to share it with the world — a library that simplifies spreadsheets, or a small app that automates weekly reports. How do you get that code into other people’s hands so they can install it and even contribute back?

That is the job of **package publishing** and **deployment**: turning a local project into something anyone can access, either through the Python Package Index or as a ready-to-run application in the cloud.

> 💡 **Key insight:** Packaging is the process of describing your project (name, version, dependencies, entry points) so tools like pip can install it cleanly. The modern way to do this is with a `pyproject.toml` file.

### Modern Packaging with pyproject.toml

Let’s build this package for real. First create the project files:

In [ ]:
!mkdir -p awesome-spreadsheet/awesome_spreadsheet

In [ ]:
%%writefile awesome-spreadsheet/pyproject.toml
[build-system]
requires = ["setuptools>=61", "wheel"]
build-backend = "setuptools.build_meta"

[project]
name = "awesome-spreadsheet"
version = "0.1.0"
description = "Simple helpers for spreadsheet automation"
readme = "README.md"
authors = [{name = "Alex Developer", email = "alex@example.com"}]
requires-python = ">=3.9"
dependencies = [
    "openpyxl>=3.1",
    "pandas>=2.0",
]

[project.scripts]
sheet-summary = "awesome_spreadsheet.cli:main"

In [ ]:
%%writefile awesome-spreadsheet/README.md
# awesome-spreadsheet

Simple helpers for spreadsheet automation.

In [ ]:
%%writefile awesome-spreadsheet/awesome_spreadsheet/__init__.py
"""Simple helpers for spreadsheet automation."""
__version__ = "0.1.0"

In [ ]:
%%writefile awesome-spreadsheet/awesome_spreadsheet/cli.py
import sys

import pandas as pd


def main() -> None:
    """sheet-summary <file.csv|file.xlsx> — print a quick summary of a spreadsheet."""
    if len(sys.argv) < 2:
        print("usage: sheet-summary <file.csv|file.xlsx>")
        return
    path = sys.argv[1]
    df = pd.read_csv(path) if path.endswith(".csv") else pd.read_excel(path)
    print(f"{len(df)} rows × {len(df.columns)} columns")
    print(df.describe(include="all"))

Now build it — this creates the wheel and source distribution in `dist/`:

In [ ]:
!pip install -q build twine
!cd awesome-spreadsheet && python -m build 2>&1 | tail -n 2
!ls awesome-spreadsheet/dist

**What the file declares**

- Build backend (setuptools).
- Project metadata and version.
- Runtime dependencies.
- Optional console-script entry points.

> 💡 **Mental model:** Think of setuptools / the build backend as a packing service. You hand it your code, a list of dependencies and some labels; it produces a neat, installable package (a wheel or source archive).

## Publishing to PyPI — The Python Package Index

Once the package is built, the usual way to share it is to upload it to **PyPI** (the Python Package Index). PyPI is the app store for Python libraries: after your package is published, anyone can install it with a single `pip install` command.

**Preparation**

Create an account on pypi.org (and test.pypi.org). Choose a unique package name. Keep a long-lived API token ready for uploads.

**Upload tool**

`twine` is the standard, secure way to upload distributions. It supports both TestPyPI and the real index.

### Build & Publish Workflow

Publishing needs your own PyPI account and API token, so the upload commands are shown for reference:

```bash
# 1. Install build tools
pip install build twine

# 2. Build wheel + source distribution
python -m build

# 3. (Recommended) upload to TestPyPI first
twine upload --repository testpypi dist/*

# 4. Verify install from TestPyPI
pip install -i https://test.pypi.org/simple/ awesome-spreadsheet

# 5. When everything looks good, publish for real
twine upload dist/*

# Users can now simply run:
# pip install awesome-spreadsheet
```

What you **can** do here: check the package, install the wheel, and run its command-line tool:

In [ ]:
!twine check awesome-spreadsheet/dist/*
!pip install -q awesome-spreadsheet/dist/*.whl
!printf "name,score\nAlice,95\nBob,87\nCarol,78\n" > scores.csv
!sheet-summary scores.csv

**Good practices**

- Always try TestPyPI before the real index.
- Use API tokens, never your password in scripts.
- Bump the version for every release (semantic versioning).
- Include a clear README and license.
- Tag the release in git to keep history traceable.

## Docker & Application Deployment

When your project is a full application rather than a pure library, packaging for pip is not enough. You also need a reproducible runtime environment. **Docker** solves this by bundling the application, its dependencies, and even the operating-system layer into a portable container.

Think of a Docker image as a shipping container: once packed, it runs the same way on your laptop, a CI server, or any cloud provider that supports Docker.

### Minimal Dockerfile for a Python App

A `Dockerfile` packages the app into a container image. Docker is not available in Colab, so build and run it on your computer:

```dockerfile
# Dockerfile
FROM python:3.12-slim

WORKDIR /app

# Install dependencies first (better layer caching)
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# Copy application code
COPY . .

# Non-root user for security
RUN useradd -m appuser && chown -R appuser /app
USER appuser

EXPOSE 8000
CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"]

# Build & run
# docker build -t my-api:0.1 .
# docker run -p 8000:8000 my-api:0.1
```

**Deployment options**

- Push the image to Docker Hub / GHCR / ECR.
- Run on a VPS with Docker Compose.
- Deploy to managed platforms (AWS ECS, Google Cloud Run, Railway, Fly.io, Render…).
- Combine with CI/CD so every git tag builds and publishes a new image.

**Libraries → PyPI**

Best when other developers will import your code. Versioned, dependency-aware, installable with pip.

**Applications → Containers**

Best when you ship a running service. Guarantees the same environment everywhere and simplifies cloud deployment.

> 💡 **Big takeaway:** Packaging with `pyproject.toml` + publishing to PyPI makes your libraries available to the whole Python world. Docker turns complete applications into portable, reproducible units that run reliably anywhere. Together they form the bridge from a local experiment to a tool the world can actually use.

---

✅ **Lesson complete!** Next up: **Lesson 6.5 · Capstone: Full-Stack Project** — [watch it in OratorLab](https://oratorlab.in/course-player.html?courseId=python-for-development&subtopicId=subtopic-6-5) or open `lesson-6-5-capstone-full-stack-project.ipynb`.

Learn the full **Free Python For Development** course for free at [oratorlab.in](https://oratorlab.in/courses/python-for-development).